In [6]:
!pip install timm

In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import timm
from torchvision import transforms
from PIL import Image

# --- 1. Configuration ---
IMAGE_DIR = '/content/drive/MyDrive/cropped_images_archive'
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
EPOCHS = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Label Extraction Logic ---
def get_label_from_filename(filename):
    """
    Assumes filename format: 'anything_LABEL_index.jpg'
    Example: 'img01_melanoma_2.jpg' -> returns 'melanoma'
    """
    parts = filename.split('_')
    # If using my previous script's naming: {name}_{label}_{i}.jpg
    # The label is the second to last element
    return parts[-2]

# Create a mapping of string labels to integers (0-7)
# Run this once to see your labels
all_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
unique_labels = sorted(list(set([get_label_from_filename(f) for f in all_files])))
label_to_id = {label: i for i, label in enumerate(unique_labels)}

print(f"Detected {len(unique_labels)} classes: {label_to_id}")

# --- 3. Dataset Logic ---
class LesionFilenameDataset(Dataset):
    def __init__(self, img_dir, filenames, transform=None):
        self.img_dir = img_dir
        self.filenames = filenames
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img_path = os.path.join(self.img_dir, fname)

        # Load image
        image = Image.open(img_path).convert('RGB')

        # Parse label
        label_str = get_label_from_filename(fname)
        label_id = label_to_id[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label_id

# --- 4. Augmentation (Stronger for small 256-image set) ---
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 5. The Architecture (HRNet-W18) ---
class HRNetLesionExpert(nn.Module):
    def __init__(self, num_classes):
        super(HRNetLesionExpert, self).__init__()
        # 'hrnet_w18.ms_aug_in1k' is a very stable pre-trained version
        self.backbone = timm.create_model('hrnet_w18', pretrained=True, num_classes=0)

        # The projection layer to create your 512-D embedding
        # HRNet-W18 feature head outputs 2048 after concatenation of branches
        self.projection = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU()
        )

        self.classifier = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x, return_embedding=False):
        # Extract features (B, 2048, 7, 7) -> Global Average Pool -> (B, 2048)
        features = self.backbone.forward_features(x)
        features = torch.mean(features, dim=(2, 3))

        # 512-D Embedding
        embedding = self.projection(features)

        if return_embedding:
            return embedding

        # Final Classification
        logits = self.classifier(self.dropout(embedding))
        return logits

# --- 6. Training Execution ---
def train_model():
    # Split data 80/20 since you only have 256 images
    train_files, val_files = random_split(all_files, [0.8, 0.2])

    train_ds = LesionFilenameDataset(IMAGE_DIR, train_files, transform=train_transform)
    val_ds = LesionFilenameDataset(IMAGE_DIR, val_files, transform=train_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = HRNetLesionExpert(len(unique_labels)).to(DEVICE)

    # Label Smoothing helps with small datasets to prevent over-confidence
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f}")

    # Save for the fusion stage
    torch.save({
        'model_state_dict': model.state_dict(),
        'label_mapping': label_to_id
    }, 'hrnet_lesion_expert.pth')

if __name__ == "__main__":
    train_model()

Detected 8 classes: {'Atelectasis': 0, 'Cardiomegaly': 1, 'Effusion': 2, 'Infiltrate': 3, 'Mass': 4, 'Nodule': 5, 'Pneumonia': 6, 'Pneumothorax': 7}
Epoch 1/50 | Loss: 1.9851
Epoch 2/50 | Loss: 1.6420
Epoch 3/50 | Loss: 1.4124
Epoch 4/50 | Loss: 1.3873
Epoch 5/50 | Loss: 1.3129
Epoch 6/50 | Loss: 1.2075
Epoch 7/50 | Loss: 1.2302
Epoch 8/50 | Loss: 1.2133
Epoch 9/50 | Loss: 1.1381
Epoch 10/50 | Loss: 1.1736
Epoch 11/50 | Loss: 1.0627
Epoch 12/50 | Loss: 0.9980
Epoch 13/50 | Loss: 1.0220
Epoch 14/50 | Loss: 0.9174
Epoch 15/50 | Loss: 0.9431
Epoch 16/50 | Loss: 0.9476
Epoch 17/50 | Loss: 0.8381
Epoch 18/50 | Loss: 0.9569
Epoch 19/50 | Loss: 0.9407
Epoch 20/50 | Loss: 0.8410
Epoch 21/50 | Loss: 0.8429
Epoch 22/50 | Loss: 0.7713
Epoch 23/50 | Loss: 0.7987
Epoch 24/50 | Loss: 0.8595
Epoch 25/50 | Loss: 0.8297
Epoch 26/50 | Loss: 0.7716
Epoch 27/50 | Loss: 0.7761
Epoch 28/50 | Loss: 0.7679
Epoch 29/50 | Loss: 0.8459
Epoch 30/50 | Loss: 0.8185
Epoch 31/50 | Loss: 0.7025
Epoch 32/50 | Loss: 0.7

In [8]:
from google.colab import files

files.download(MODEL_PATH)


NameError: name 'model' is not defined